In [29]:
import os
import faiss
from dotenv import load_dotenv
from transformers import AutoTokenizer

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
    Settings,
    set_global_tokenizer,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.vector_stores.faiss import FaissVectorStore

In [30]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [31]:
set_global_tokenizer(AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5").encode)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.chunk_size = 450
Settings.chunk_overlap = 50

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [32]:
documents = SimpleDirectoryReader(
    input_dir="/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data",
    exclude=[
        "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/README.md",
        "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/paul_graham_essay.txt",
    ],
).load_data(num_workers=4)

In [33]:
print(f"Loaded {len(documents)} documents.")

Loaded 11 documents.


In [34]:
documents[0].metadata

{'file_path': '/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/10min.rst',
 'file_name': '10min.rst',
 'file_type': 'text/x-rst',
 'file_size': 18933,
 'creation_date': '2026-07-21',
 'last_modified_date': '2026-07-21'}

In [35]:
d = 384
faiss_index = faiss.IndexFlatL2(d)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# from local drives 
vector_store_local = FaissVectorStore.from_persist_dir(
    "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Storage"
)
storage_context_local = StorageContext.from_defaults(
    persist_dir="/Users/mohitag/Documents/Projects/RAG_SQL_Project/Storage",
    vector_store=vector_store_local,
)
index_local = load_index_from_storage(storage_context_local)

In [39]:
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, show_progress=True
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/717 [00:00<?, ?it/s]

In [40]:
print(len(index.docstore.docs))

717


In [41]:
for node_id, node in list(index.docstore.docs.items())[:5]:
    print("--- CHUNK ---")
    print("Source file:", node.metadata.get("file_name"))
    print("Text preview:", node.text[:300])
    print()

--- CHUNK ---
Source file: 10min.rst
Text preview: .. _10min:

{{ header }}

********************
10 minutes to pandas
********************

This is a short introduction to pandas, geared mainly for new users.
You can see more complex recipes in the :ref:`Cookbook<cookbook>`.

Customarily, we import as follows:

.. ipython:: python

   import numpy 

--- CHUNK ---
Source file: 10min.rst
Text preview: Creating a :class:`Series` by passing a list of values, letting pandas create
a default :class:`RangeIndex`.

.. ipython:: python

   s = pd.Series([1, 3, 5, np.nan, 6, 8])
   s

Creating a :class:`DataFrame` by passing a NumPy array with a datetime index using :func:`date_range`
and labeled columns

--- CHUNK ---
Source file: 10min.rst
Text preview: Here's a subset of the attributes that
will be completed:

.. ipython::

   @verbatim
   In [1]: df2.<TAB>  # noqa: E225, E999
   df2.A                  df2.bool
   df2.abs                df2.boxplot
   df2.add                df2.C
   df2.add_

In [42]:
for node_id, node in list(index.docstore.docs.items())[:5]:
    print(len(node.text), "characters")

918 characters
1200 characters
804 characters
879 characters
1004 characters


In [43]:
for node_id, node in list(index.docstore.docs.items())[:12]:
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
    print(len(tok.encode(node.text)), "tokens")

297 tokens
398 tokens
263 tokens
285 tokens
325 tokens
302 tokens
259 tokens
242 tokens
408 tokens
270 tokens
352 tokens
384 tokens


In [44]:
print(index.ref_doc_info)

{'308be2fb-7631-4a47-8b28-56c7d339a8c3': RefDocInfo(node_ids=['1b19ce6e-ede0-40aa-b318-8e9f9b3d8cca', 'efc38f40-acf0-4885-9617-396db49a7563', 'a09db7c3-e8ca-4f7e-990d-9a9ece94b56b', '1cd1c3b5-1225-48f6-af1f-48246d2ddee0', '2e473dde-6566-44db-8d78-f01b3dfdbc15', '2f7148dd-ae31-443f-b149-e8a32b6e6ec1', '8514692d-ef0d-48ea-93e1-9f412566fc94', 'ba18549c-39ec-46b0-a559-6c08854fd75f', '5075957b-6bf9-4525-9869-ccb995bb907d', '4c4d48a3-b32c-446e-b1a1-886b5725eaf5', 'b2347df1-c246-48e2-af33-372e3918c22c', '4c183067-0cb8-4ed1-932b-7a485b366ba5', '9060ade3-17c6-4158-a2be-73087e5b7df1', '4ae7dc43-2c06-4fc0-89ae-453b0035231c', '0dd622f5-9da7-4cff-b8e3-2ed152337e60', '5ef9e310-23ab-478e-b9a6-ad5d1f551108', '9434c997-8b16-4b74-947c-b6074995b9ce', '218996bb-6cf8-4e5c-a31c-c02001e1987e', 'c112cd1b-fd7e-43be-907d-0356116781d5', 'd44ed2dc-079b-48ab-a53a-b00ecda468ae', '22d9f03a-9ec9-457f-974c-e4afb5ad63e0', '0a7ae39b-5690-49bd-b52f-31fcc16a7e8e'], metadata={'file_path': '/Users/mohitag/Documents/Projects

In [45]:
print(documents[0].text)

.. _10min:

{{ header }}

********************
10 minutes to pandas
********************

This is a short introduction to pandas, geared mainly for new users.
You can see more complex recipes in the :ref:`Cookbook<cookbook>`.

Customarily, we import as follows:

.. ipython:: python

   import numpy as np
   import pandas as pd

Basic data structures in pandas
-------------------------------

pandas provides two types of classes for handling data:

1. :class:`Series`: a one-dimensional labeled array holding data of any type
    such as integers, strings, Python objects etc.
2. :class:`DataFrame`: a two-dimensional data structure that holds data like
   a two-dimension array or a table with rows and columns.

Object creation
---------------

See the :ref:`Intro to data structures section <dsintro>`.

Creating a :class:`Series` by passing a list of values, letting pandas create
a default :class:`RangeIndex`.

.. ipython:: python

   s = pd.Series([1, 3, 5, np.nan, 6, 8])
   s

Creating a 

In [46]:
Settings.llm = HuggingFaceInferenceAPI(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    token=hf_token,
    temperature=0.2,
    max_tokens=256,
    provider="auto",
)

In [47]:
query_engine = index.as_query_engine()

In [48]:
response = query_engine.query("how to create a csv file?")

In [49]:
print(response.response)

To create a CSV file, you can use the `to_csv` method of a Pandas DataFrame or Series. This method allows you to store the contents of the object as a comma-separated-values file. The required argument is the file path where you want to save the CSV file. You can also specify additional parameters to customize the CSV file, such as the index, header, and separator. For example:

```
df = pd.DataFrame(np.random.randint(0, 5, (10, 5)))
df.to_csv("example.csv", index=False)
```

This will create a CSV file named "example.csv" in the current working directory, with the DataFrame's contents stored in it. The `index=False` argument tells Pandas not to include the index column in the CSV file.


In [50]:
response = query_engine.query("create a dataframe?")

In [51]:
print(response.response)

Here's a simple example of creating a DataFrame:

```python
import pandas as pd
import numpy as np

# Create a dictionary with data
data = {
    "Name": ["Tom", "Nick", "John"],
    "Age": [20, 21, 19],
    "Score": [90, 85, 88]
}

# Create a DataFrame
df = pd.DataFrame(data)

# Print the DataFrame
print(df)
```


In [52]:
query_engine_local = index_local.as_query_engine()


In [57]:
print(query_engine_local.query("load dataframe from cloud with examples").response)

To load a DataFrame from the cloud, you can use the `read_csv` function from the pandas library, which allows you to read a CSV file from a URL. For example, if you have a CSV file hosted on AWS S3, you can use the `read_csv` function with the `storage_options` parameter to specify the AWS S3 credentials. Here's an example:

```python
import pandas as pd

# Specify the URL of the CSV file on AWS S3
url = 'https://s3.amazonaws.com/your-bucket-name/your-file.csv'

# Specify the AWS S3 credentials
storage_options = {'anon': False, 'aws_access_key_id': 'YOUR_ACCESS_KEY',
                   'aws_secret_access_key': 'YOUR_SECRET_KEY'}

# Load the DataFrame from the CSV file on AWS S3
df = pd.read_csv(url, storage_options=storage_options)

# Display the loaded DataFrame
print(df)
```

Alternatively, if you have a CSV file hosted on Google Cloud Storage, you can use the `read_csv` function with the `storage_options` parameter to specify the GCS credentials. Here's an example:

```python
import